In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00


In [ ]:
!unzip -q -o tika-core.zip -d /content/

In [ ]:
import os
import torch
import json
from pathlib import Path
from collections import defaultdict
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
import gc
import csv
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
alpha = 0.4
k = 12

source_files = Path("/content/tika-core/src/main/java")

# Raw RSF paths
arc_rsf_path = Path(f"/content/tikadp-v1_ARC_{k}_clusters_alpha={alpha}.rsf")
acdc_rsf_path = Path("/content/tika_detect_parser_acdc.rsf")
limbo_rsf_path = Path(f"/content/tikadp-v1_IL_{k}_clusters.rsf")

selected_algorithm = "LIMBO"   # Change manually when needed

RUN_CONFIGS = {
    "ARC": {
        "algorithm_name": f"ARC_alpha_{alpha}_k_{k}",
        "rsf_path": arc_rsf_path,
        "output_dir": Path(f"/content/week4_summary_ARC_alpha_{alpha}_k_{k}"),
        "preprocess_inner_classes": False
    },

    "ACDC": {
        "algorithm_name": "ACDC",
        "rsf_path": acdc_rsf_path,
        "output_dir": Path("/content/week4_summary_ACDC"),
        "preprocess_inner_classes": True
    },

    "LIMBO": {
        "algorithm_name": f"LIMBO_k_{k}",
        "rsf_path": limbo_rsf_path,
        "output_dir": Path(f"/content/week4_summary_LIMBO_k_{k}"),
        "preprocess_inner_classes": True
    }
}

run_config = RUN_CONFIGS[selected_algorithm]
algorithm_name = run_config["algorithm_name"]
rsf_path = run_config["rsf_path"]
output_dir = run_config["output_dir"]
preprocess_inner_classes = run_config["preprocess_inner_classes"]

output_dir.mkdir(parents=True, exist_ok=True)

print("Selected algorithm:", selected_algorithm)
print("Algorithm name:", algorithm_name)
print("RSF path:", rsf_path)
print("Output directory:", output_dir)
print("Preprocess inner classes:", preprocess_inner_classes)

if not rsf_path.exists():
    print(f"WARNING: RSF file does not exist: {rsf_path}")

Selected algorithm: LIMBO
Algorithm name: LIMBO_k_12
RSF path: /content/tikadp-v1_IL_12_clusters.rsf
Output directory: /content/week4_summary_LIMBO_k_12
Preprocess inner classes: True


In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

In [ ]:
def normalize_to_file_level_class(class_name):
    """
    Convert inner-class names to outer/source-file class names.

    Example:
    org.apache.tika.parser.NetworkParser$ParsingTask
    -> org.apache.tika.parser.NetworkParser
    """
    return class_name.split("$")[0]

def get_clusters(rsf_file, preprocess_inner_classes=False):
    """
    Read RSF file and group classes by cluster ID.

    If preprocess_inner_classes=True:
    - Convert inner-class entries to their outer class.
    - Remove duplicate outer classes within each cluster.
    """
    clusters = defaultdict(set)

    if not Path(rsf_file).exists():
        print(f"ERROR: RSF file not found: {rsf_file}")
        return {}

    with open(rsf_file, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) == 3 and parts[0] == "contain":
                cluster_id = parts[1]
                class_name = parts[2]

                if preprocess_inner_classes:
                    class_name = normalize_to_file_level_class(class_name)

                clusters[cluster_id].add(class_name)

    return {
        cluster_id: sorted(list(class_names))
        for cluster_id, class_names in clusters.items()
    }


def cluster_sort_key(item):
    """
    Sort numeric cluster IDs numerically and string cluster IDs alphabetically.
    Works for ARC/LIMBO numeric IDs and ACDC string IDs.
    """
    cluster_id = item[0]

    try:
        return (0, int(cluster_id))
    except ValueError:
        return (1, str(cluster_id))


def count_inner_class_entries(clusters):
    """Count how many class names still contain '$'."""
    return sum(
        1
        for class_list in clusters.values()
        for class_name in class_list
        if "$" in class_name
    )


def save_clusters_to_rsf(clusters, output_path):
    """
    Save preprocessed clusters back to an RSF file.
    Format:
    contain cluster_id class_name
    """
    with open(output_path, "w", encoding="utf-8") as f:
        for cluster_id, class_list in sorted(clusters.items(), key=cluster_sort_key):
            for class_name in class_list:
                f.write(f"contain {cluster_id} {class_name}\n")

    print(f"Saved file-based RSF to: {output_path}")


def get_source_file(full_class_name, source_root):
    parts = full_class_name.split(".")
    parts[-1] = parts[-1].split("$")[0]  # handle inner classes
    rel_path = Path(*parts).with_suffix(".java")

    full_path = source_root / rel_path
    if full_path.exists() and full_path.is_file():
        try:
            return full_path.read_text(encoding="utf-8", errors="replace"), str(full_path)
        except Exception as e:
            print(f"Error reading {full_path}: {e}")
            return None, None

    return None, None

def clean_csv_cell(text):
    """Remove newlines and excessive whitespace for clean CSV cells."""
    if text is None:
        return ""
    return " ".join(str(text).split())


def export_summary_csv(all_reports, output_path):
    """Export Week 4 results to CSV format per assignment requirements."""
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        writer.writerow(["cluster_ID", "files", "title", "description"])

        for cluster_id, report in sorted(all_reports.items(), key=cluster_sort_key):
            files = clean_csv_cell("|".join(report.get("files", [])))
            title = clean_csv_cell(report.get("title", "Untitled Cluster"))
            description = clean_csv_cell(report.get("description", ""))

            writer.writerow([
                cluster_id,
                files,
                title,
                description
            ])

    print(f"CSV exported to: {output_path}")

def parse_cluster_result(text):
    """Parse LLM cluster output into title and description."""
    try:
        data = json.loads(text)
    except Exception:
        try:
            start = text.index("{")
            end = text.rindex("}") + 1
            data = json.loads(text[start:end])
        except Exception:
            cleaned = clean_csv_cell(text)
            return "Untitled Cluster", cleaned

    title = clean_csv_cell(data.get("title", "Untitled Cluster"))
    description = clean_csv_cell(data.get("description", ""))

    if not title:
        title = "Untitled Cluster"
    if not description:
        description = "Description unavailable"

    return title, description

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
print(f"\nLoading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    token=hf_token
)

def generate_summary(system_prompt, user_content, max_tokens=512):
    """Run Qwen generation with your chosen parameters."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.5,
            top_p=0.8,
            do_sample=True
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs.input_ids, outputs)
    ]

    del inputs
    del outputs
    torch.cuda.empty_cache()
    gc.collect()

    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

FILE_SYSTEM_PROMPT = """
You are an expert software architect analyzing Java source code.
Generate a concise structured summary with these headings:
1. Key Functionality
2. Core Logic
3. Inputs/Outputs
4. Dependencies
5. Role in Apache Tika Detect/Parser subsystem

Keep the summary factual and grounded in the provided code only.
"""

PACKAGE_SYSTEM_PROMPT = """
You are an expert software architect.
Based ONLY on the provided file summaries, generate a concise package/submodule summary with:
1. Module Title
2. Package/Submodule Role
3. Shared Responsibilities
4. Internal Interactions
5. Important Dependencies
6. Technologies/Frameworks/Tools Mentioned

Do not invent details not supported by the summaries.
Keep the summary concise.
"""

CLUSTER_SYSTEM_PROMPT = """
You are an expert software architect.

Based ONLY on the provided package/module summaries, generate the final cluster-level architectural recovery result.

Return ONLY valid JSON with exactly these two fields:

{
  "title": "A short LLM-generated architectural title for the cluster, under 8 words.",
  "description": "A concise high-level summary under 150 words. It must include: components and interactions, quality attributes such as scalability/security/maintainability/performance/reliability, and technologies/frameworks/languages/tools used."
}

Rules:
- Do not use markdown.
- Do not include bullet points.
- Do not include raw code.
- Do not invent unsupported details.
- The description must be under 150 words.
"""




Loading model: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
if __name__ == "__main__":
  print(f"\nStarting summarization for {algorithm_name}: {rsf_path.name}")
  print("Preprocess inner classes:", preprocess_inner_classes)
  clusters = get_clusters(
      rsf_path,
      preprocess_inner_classes=preprocess_inner_classes
  )
  if preprocess_inner_classes:
    save_clusters_to_rsf(clusters, output_dir / f"{algorithm_name}_file_based.rsf")
  print(f"Total clusters found: {len(clusters)}")
  all_reports = {}
  missing_files = []

  for cluster_id, class_list in sorted(clusters.items(), key=cluster_sort_key):
      print(f"\n--- Processing Cluster {cluster_id} ({len(class_list)} classes) ---")
      print(f"  [Cluster] Files included: {', '.join(class_list)}")

      file_summaries = {}
      package_groups = defaultdict(list)

      # ------------------------------------------
      # STEP A: FILE-LEVEL SUMMARIES
      # ------------------------------------------
      for cls_name in tqdm(class_list, desc=f"Cluster {cluster_id} files"):
          raw_code, file_path = get_source_file(cls_name, source_files)

          if raw_code is None:
              print(f"  [Skip] Source file not found for {cls_name}")
              missing_files.append(cls_name)
              continue

          truncated_code = raw_code[:4500]   # simple safety truncation

          user_prompt = f"Analyze this Java class: {cls_name}\nFile path: {file_path}\n\n```java\n{truncated_code}\n```"
          print(f"  [File] Generating summary for: {cls_name}")
          summary = generate_summary(FILE_SYSTEM_PROMPT, user_prompt, max_tokens=500)

          file_summaries[cls_name] = {
              "file_path": file_path,
              "summary": summary
          }

          package_name = ".".join(cls_name.split(".")[:-1])
          package_groups[package_name].append(f"Class: {cls_name}\nSummary:\n{summary}\n")

      # ------------------------------------------
      # STEP B: PACKAGE-LEVEL SUMMARIES
      # ------------------------------------------
      package_summaries = {}

      for pkg_name, summaries in package_groups.items():
          print(f"  [Package] Summarizing directory: {pkg_name} ({len(summaries)} files)")
          package_class_names = [
              s.split("Class:")[1].splitlines()[0].strip()
              for s in summaries
          ]
          print(f"  [Package] Files included: {', '.join(package_class_names)}")

          pkg_text = "\n".join(summaries)[:9000]

          user_prompt = f"Here are file summaries for the Java package/submodule '{pkg_name}':\n\n{pkg_text}"
          pkg_summary = generate_summary(PACKAGE_SYSTEM_PROMPT, user_prompt, max_tokens=600)

          package_summaries[pkg_name] = pkg_summary

      # ------------------------------------------
      # STEP C: CLUSTER-LEVEL SUMMARY
      # ------------------------------------------
      print(f"  [Cluster] Generating final architecture summary for Cluster {cluster_id}")
      print(f"  [Cluster] Packages included: {', '.join(package_summaries.keys())}")

      aggregated_packages = ""
      for pkg_name, pkg_summary in package_summaries.items():
          aggregated_packages += f"\nPackage: {pkg_name}\nSummary:\n{pkg_summary}\n"

      aggregated_packages = aggregated_packages[:11000]

      user_prompt = f"""Here are the package/module summaries for Cluster {cluster_id}:\n
      {aggregated_packages}\n
      Generate the TITLE and DESCRIPTION for Cluster {cluster_id}.
      """
      cluster_summary_raw = generate_summary(CLUSTER_SYSTEM_PROMPT, user_prompt, max_tokens=350)
      title, description = parse_cluster_result(cluster_summary_raw)

      cluster_file_paths = sorted({
          info["file_path"]
          for info in file_summaries.values()
          if "file_path" in info
      })

      cluster_report = {
          "algorithm": algorithm_name,
          "cluster_id": cluster_id,
          "classes": class_list,
          "files": cluster_file_paths,
          "file_summaries": file_summaries,
          "package_summaries": package_summaries,
          "title": title,
          "description": description,
          "architecture_summary_raw": cluster_summary_raw
      }

      all_reports[cluster_id] = cluster_report

      # Save per-cluster immediately
      safe_cluster_id = str(cluster_id).replace("/", "_").replace("\\", "_").replace(":", "_")
      with open(output_dir / f"cluster_{safe_cluster_id}_summary.json", "w", encoding="utf-8") as f:
          json.dump(cluster_report, f, indent=2, ensure_ascii=False)

  with open(output_dir / "missing_files.json", "w", encoding="utf-8") as f:
      json.dump(missing_files, f, indent=2, ensure_ascii=False)

  # Save combined results
  with open(output_dir / "all_cluster_reports.json", "w", encoding="utf-8") as f:
      json.dump(all_reports, f, indent=2, ensure_ascii=False)

  print("\n✅ Week 4 summarization complete.")
  print("Saved results to:", output_dir)

  export_summary_csv(
      all_reports,
      output_dir / f"cluster_summary_{algorithm_name}.csv"
  )

  zip_name = f"/content/week4_summary_{algorithm_name}.zip"
  !zip -q -r "{zip_name}" "{output_dir}"

  from google.colab import files
  files.download(zip_name)


Starting summarization for LIMBO_k_12: tikadp-v1_IL_12_clusters.rsf
Preprocess inner classes: True
Saved file-based RSF to: /content/week4_summary_LIMBO_k_12/LIMBO_k_12_file_based.rsf
Total clusters found: 12

--- Processing Cluster 0 (5 classes) ---
  [Cluster] Files included: org.apache.tika.Tika, org.apache.tika.extractor.EmbeddedDocumentUtil, org.apache.tika.extractor.ParserContainerExtractor, org.apache.tika.mime.MimeTypes, org.apache.tika.mime.ProbabilisticMimeDetectionSelector


Cluster 0 files:   0%|          | 0/5 [00:00<?, ?it/s]

  [File] Generating summary for: org.apache.tika.Tika


Cluster 0 files:  20%|██        | 1/5 [00:35<02:22, 35.72s/it]

  [File] Generating summary for: org.apache.tika.extractor.EmbeddedDocumentUtil


...

  [File] Generating summary for: org.apache.tika.mime.ProbabilisticMimeDetectionSelector


Cluster 0 files: 100%|██████████| 5/5 [03:12<00:00, 38.55s/it]


  [Package] Summarizing directory: org.apache.tika (1 files)
  [Package] Files included: org.apache.tika.Tika
  [Package] Summarizing directory: org.apache.tika.extractor (2 files)
  [Package] Files included: org.apache.tika.extractor.EmbeddedDocumentUtil, org.apache.tika.extractor.ParserContainerExtractor
  [Package] Summarizing directory: org.apache.tika.mime (2 files)
  [Package] Files included: org.apache.tika.mime.MimeTypes, org.apache.tika.mime.ProbabilisticMimeDetectionSelector
  [Cluster] Generating final architecture summary for Cluster 0
  [Cluster] Packages included: org.apache.tika, org.apache.tika.extractor, org.apache.tika.mime

--- Processing Cluster 1 (37 classes) ---
  [Cluster] Files included: org.apache.tika.config.ConfigDeserializer, org.apache.tika.config.JsonConfig, org.apache.tika.config.SelfConfiguring, org.apache.tika.config.TikaComponent, org.apache.tika.detect.CharsetSupersets, org.apache.tika.detect.TextStatistics, org.apache.tika.detect.TrainedModel, org.ap

Cluster 1 files:   0%|          | 0/37 [00:00<?, ?it/s]

  [File] Generating summary for: org.apache.tika.config.ConfigDeserializer
